# Sprint 5 — Omralinov-inspired pipeline for S6E8 Smartphone Addiction

**Kaggle Playground Series S6E8** | ROC AUC | Target: LB 0.970+

## Key innovations from Tamerlan Omralinov (0.97052 public, rank 353)
1. **Budget constraint features** (other_screen, other_frac) — +0.00096/fold ablation
2. **Exact-value embedding tables** per column (lookup signal from quantized data)
3. **PLR (Periodic-Linear Representation)** — learned Fourier + linear for smooth trends
4. **Transformer attention** on feature tokens for interactions
5. **NaN as index 0** → learned embedding per column
6. **Random masking** during training (augmentation on missingness patterns)
7. **11 folds, 32 epochs, AdamW + OneCycle + EMA**

**Companions**: CatBoost (numeric+categorical) + LightGBM (target-encoding)
**Blend**: NN 0.49 / CatBoost 0.31 / LightGBM 0.21 on rank-normalized OOF

---
*Sprint 5 — Omralinov-inspired pipeline for S6E8 Smartphone Addiction*

In [ ]:
"""
Sprint 5 — Omralinov-inspired pipeline for S6E8 Smartphone Addiction
=====================================================================
Kaggle kernel version — runs on free P100 GPU.

Key innovations from Tamerlan Omralinov (0.97052 public, rank 353):
1. Budget constraint features (other_screen, other_frac) — +0.00096/fold ablation
2. Exact-value embedding tables per column (lookup signal from quantized data)
3. PLR (Periodic-Linear Representation) — learned Fourier + linear for smooth trends
4. Transformer attention on feature tokens for interactions
5. NaN as index 0 → learned embedding per column
6. Random masking during training (augmentation on missingness patterns)
7. 11 folds, 32 epochs, AdamW + OneCycle + EMA

Companions: CatBoost (numeric+categorical) + LightGBM (target-encoding)
Blend: NN 0.49 / CatBoost 0.31 / LightGBM 0.21 on rank-normalized OOF
"""

import numpy as np
import pandas as pd
import os
import json
import time
import warnings
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

# ── Feature columns ────────────────────────────────────────────────────────
NUM_COLS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time'
]
CAT_COLS = ['gender', 'stress_level', 'academic_work_impact']
TARGET = 'addicted_label'
BUDGET_TOTAL = 'daily_screen_time_hours'
BUDGET_COMPONENTS = ['social_media_hours', 'gaming_hours', 'work_study_hours']


# ═══════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING & FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════════════════

def load_data():
    """Search ALL /kaggle/input/ subdirectories for competition data."""
    import glob
    # Comprehensive search: direct, one-level, two-level, and full recursive
    patterns = [
        "/kaggle/input/train.csv",
        "/kaggle/input/*/train.csv",
        "/kaggle/input/*/*/train.csv",
        "/kaggle/input/*/*/*/train.csv",
    ]
    for pattern in patterns:
        found = glob.glob(pattern)
        if found:
            base = os.path.dirname(found[0])
            # Verify all 3 files exist
            if (os.path.exists(os.path.join(base, "test.csv")) and
                os.path.exists(os.path.join(base, "sample_submission.csv"))):
                print(f"Found data at: {base}")
                train = pd.read_csv(os.path.join(base, "train.csv"))
                test = pd.read_csv(os.path.join(base, "test.csv"))
                sub = pd.read_csv(os.path.join(base, "sample_submission.csv"))
                return train, test, sub
    # Fallback: full recursive walk
    for root, dirs, files in os.walk("/kaggle/input/"):
        if "train.csv" in files and "test.csv" in files and "sample_submission.csv" in files:
            print(f"Found data at: {root}")
            train = pd.read_csv(os.path.join(root, "train.csv"))
            test = pd.read_csv(os.path.join(root, "test.csv"))
            sub = pd.read_csv(os.path.join(root, "sample_submission.csv"))
            return train, test, sub
    # Last resort: list everything for debugging
    print("Contents of /kaggle/input/:")
    for root, dirs, files in os.walk("/kaggle/input/"):
        for f in files:
            print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError("No train.csv found anywhere in /kaggle/input/")


def add_budget_features(df):
    """Finding 2: Generative constraint — daily_screen >= social + gaming + work.
    The residual 'unaccounted screen time' is a strong signal (AUC 0.765).
    GBDTs cannot construct this 4-term sum with axis-aligned splits."""
    total = df[BUDGET_TOTAL].fillna(0)
    components_sum = df[BUDGET_COMPONENTS].fillna(0).sum(axis=1)
    df['other_screen'] = total - components_sum
    df['other_frac'] = df['other_screen'] / df[BUDGET_TOTAL].replace(0, np.nan)
    df['other_frac'] = df['other_frac'].fillna(0).clip(0, 1)
    df['on_boundary'] = (df['other_screen'] == 0).astype(float)
    for comp in BUDGET_COMPONENTS:
        df[f'{comp}_frac'] = df[comp].fillna(0) / df[BUDGET_TOTAL].replace(0, np.nan)
        df[f'{comp}_frac'] = df[f'{comp}_frac'].fillna(0).clip(0, 1)
    return df


def add_frequency_features(df, fit_df=None):
    """Value frequency features — fit on train only."""
    for col in NUM_COLS:
        if fit_df is not None:
            freq = fit_df[col].value_counts(normalize=True)
        else:
            freq = df[col].value_counts(normalize=True)
        df[f'{col}_freq'] = df[col].map(freq).fillna(0)
    return df


def add_missingness_features(df):
    """Missingness indicators — the NaN pattern carries signal."""
    for col in NUM_COLS:
        df[f'{col}_missing'] = df[col].isna().astype(float)
    df['n_missing'] = df[[f'{c}_missing' for c in NUM_COLS]].sum(axis=1)
    return df


def prepare_data(train, test):
    """Full feature engineering pipeline."""
    train = add_budget_features(train)
    test = add_budget_features(test)
    train = add_frequency_features(train)
    test = add_frequency_features(test, fit_df=train)
    train = add_missingness_features(train)
    test = add_missingness_features(test)

    # Fix CAT_COLS NaN for CatBoost: convert NaN to 'missing' string BEFORE any model sees it
    # astype(str) on str dtype keeps NaN as NaN, so we must fillna first
    for col in CAT_COLS:
        train[col] = train[col].fillna('missing').astype(str)
        test[col] = test[col].fillna('missing').astype(str)

    # Exact-value categorical columns (used by CatBoost)
    # Round to 2 decimals before str conversion to match vocab quantization and ensure
    # consistent string representation (e.g. '3.5' not '3.50')
    for col in NUM_COLS:
        ec = f'{col}__exact_cat'
        rounded_train = train[col].round(2)
        rounded_test = test[col].round(2)
        train[ec] = rounded_train.astype(str)
        train.loc[train[col].isna(), ec] = 'missing'
        test[ec] = rounded_test.astype(str)
        test.loc[test[col].isna(), ec] = 'missing'

    return train, test


# ═══════════════════════════════════════════════════════════════════════════
# 2. CATBOOST — dual numeric + exact-value categorical
# ═══════════════════════════════════════════════════════════════════════════

def train_catboost(train, test, n_folds=11, seed=42):
    from catboost import CatBoostClassifier, Pool

    feature_cols = NUM_COLS + CAT_COLS + [
        'other_screen', 'other_frac', 'on_boundary',
        'social_media_hours_frac', 'gaming_hours_frac', 'work_study_hours_frac',
        'n_missing'
    ] + [f'{c}_freq' for c in NUM_COLS]

    # Exact-value categorical columns (per-column, created in prepare_data to avoid re-computation)
    exact_cat_cols = [f'{c}__exact_cat' for c in NUM_COLS]
    all_features = feature_cols + exact_cat_cols
    cat_features_idx = [all_features.index(c) for c in CAT_COLS + exact_cat_cols]

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train[TARGET])):
        t0 = time.time()
        X_tr = train.iloc[tr_idx][all_features]
        y_tr = train.iloc[tr_idx][TARGET]
        X_va = train.iloc[va_idx][all_features]
        y_va = train.iloc[va_idx][TARGET]
        X_te = test[all_features]

        model = CatBoostClassifier(
            iterations=6000,
            learning_rate=0.03,
            depth=8,
            l2_leaf_reg=3.0,
            random_seed=seed + fold,
            eval_metric='AUC',
            verbose=0,
            early_stopping_rounds=200,
            cat_features=cat_features_idx,
            one_hot_max_size=50,
            task_type='GPU',
            devices='0',
        )

        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1] / n_folds

        fold_auc = roc_auc_score(y_va, oof[va_idx])
        fold_aucs.append(fold_auc)
        elapsed = time.time() - t0
        print(f"  CatBoost fold {fold+1}/{n_folds}: AUC={fold_auc:.6f} ({elapsed:.1f}s)")

    overall_auc = roc_auc_score(train[TARGET], oof)
    print(f"  CatBoost OOF AUC: {overall_auc:.6f}")
    return oof, test_preds, {'overall': overall_auc, 'folds': fold_aucs}


# ═══════════════════════════════════════════════════════════════════════════
# 3. LIGHTGBM — fold-safe target encoding
# ═══════════════════════════════════════════════════════════════════════════

def train_lightgbm(train, test, n_folds=11, seed=42):
    import lightgbm as lgb

    feature_cols = NUM_COLS + CAT_COLS + [
        'other_screen', 'other_frac', 'on_boundary',
        'social_media_hours_frac', 'gaming_hours_frac', 'work_study_hours_frac',
        'n_missing'
    ] + [f'{c}_freq' for c in NUM_COLS]

    # Target encoding columns
    te_cols = NUM_COLS.copy()
    for col in te_cols:
        train[f'{col}__te'] = np.nan
        test[f'{col}__te'] = 0.0

    te_feature_cols = [f'{c}__te' for c in te_cols]
    all_features = feature_cols + te_feature_cols

    # Encode categoricals — fit on combined train+test to avoid unseen category issues.
    # Work on local copies so we don't mutate the caller's DataFrames (NN runs after this).
    train = train.copy()
    test = test.copy()
    for col in CAT_COLS:
        all_vals = pd.concat([train[col], test[col]]).astype('category')
        train[col] = all_vals.iloc[:len(train)].cat.codes.astype(np.int32)
        test[col] = all_vals.iloc[len(train):].cat.codes.astype(np.int32)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train[TARGET])):
        t0 = time.time()

        # Fold-safe target encoding
        for col in te_cols:
            global_mean = train.iloc[tr_idx][TARGET].mean()
            te_map = train.iloc[tr_idx].groupby(col)[TARGET].agg(['mean', 'count'])
            smoothing = 10
            te_map['smoothed'] = (
                te_map['count'] * te_map['mean'] + smoothing * global_mean
            ) / (te_map['count'] + smoothing)
            te_dict = te_map['smoothed'].to_dict()
            train.loc[train.index[va_idx], f'{col}__te'] = (
                train.iloc[va_idx][col].map(te_dict).fillna(global_mean)
            )
            test[f'{col}__te'] += test[col].map(te_dict).fillna(global_mean) / n_folds

        X_tr = train.iloc[tr_idx][all_features]
        y_tr = train.iloc[tr_idx][TARGET]
        X_va = train.iloc[va_idx][all_features]
        y_va = train.iloc[va_idx][TARGET]

        model = lgb.LGBMClassifier(
            n_estimators=6000,
            learning_rate=0.03,
            num_leaves=63,
            max_depth=7,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=seed + fold,
            verbose=-1,
            n_jobs=-1,
        )

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(test[all_features])[:, 1] / n_folds

        fold_auc = roc_auc_score(y_va, oof[va_idx])
        fold_aucs.append(fold_auc)
        elapsed = time.time() - t0
        print(f"  LightGBM fold {fold+1}/{n_folds}: AUC={fold_auc:.6f} ({elapsed:.1f}s)")

    overall_auc = roc_auc_score(train[TARGET], oof)
    print(f"  LightGBM OOF AUC: {overall_auc:.6f}")
    return oof, test_preds, {'overall': overall_auc, 'folds': fold_aucs}


# ═══════════════════════════════════════════════════════════════════════════
# 4. NEURAL NET — Embedding + PLR + Transformer (Omralinov-inspired)
# ═══════════════════════════════════════════════════════════════════════════

def train_neural_net(train, test, n_folds=11, seed=42, epochs=32):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  NN device: {device}")

    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Build vocabulary for each numeric column (round to 2 decimals to match quantization)
    vocab_maps = {}
    for col in NUM_COLS:
        vals = np.sort(np.round(train[col].dropna().unique(), 2))
        vocab = {v: i + 1 for i, v in enumerate(vals)}
        vocab_maps[col] = vocab

    budget_cols = ['other_screen', 'other_frac', 'on_boundary',
                   'social_media_hours_frac', 'gaming_hours_frac', 'work_study_hours_frac']
    missing_cols = [f'{c}_missing' for c in NUM_COLS] + ['n_missing']
    n_numeric = len(budget_cols) + len(missing_cols)

    class EmbeddingPLRTransformer(nn.Module):
        def __init__(self, vocab_sizes, embed_dim=16, n_heads=4, n_layers=2,
                     plr_frequencies=16, dropout=0.1):
            super().__init__()
            self.n_features = len(vocab_sizes)
            self.embed_dim = embed_dim

            # Per-column embedding tables (index 0 = NaN/masked)
            self.embeddings = nn.ModuleList([
                nn.Embedding(vs + 1, embed_dim, padding_idx=0)
                for vs in vocab_sizes
            ])

            # PLR: learned periodic (Fourier) + linear
            self.plr_linear = nn.ModuleList([
                nn.Linear(1, embed_dim) for _ in range(self.n_features)
            ])
            self.plr_freq = nn.ParameterList([
                nn.Parameter(torch.randn(plr_frequencies))
                for _ in range(self.n_features)
            ])
            self.plr_phase = nn.ParameterList([
                nn.Parameter(torch.randn(plr_frequencies))
                for _ in range(self.n_features)
            ])
            self.plr_fourier_proj = nn.ModuleList([
                nn.Linear(plr_frequencies * 2, embed_dim)
                for _ in range(self.n_features)
            ])

            # Budget/missing → 2 tokens
            self.numeric_proj = nn.Linear(n_numeric, embed_dim * 2)

            # Type embedding
            self.type_embed = nn.Embedding(3, embed_dim)

            # Transformer
            n_tokens = self.n_features * 2 + 2
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 4,
                dropout=dropout, batch_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

            # Head
            self.head = nn.Sequential(
                nn.LayerNorm(embed_dim * n_tokens),
                nn.Linear(embed_dim * n_tokens, 128),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(128, 1)
            )

        def forward(self, cat_idx, numeric_vals, budget_numeric, mask):
            B = cat_idx.size(0)
            tokens = []

            for i in range(self.n_features):
                # Embedding token
                emb = self.embeddings[i](cat_idx[:, i])

                # PLR token
                x = numeric_vals[:, i:i+1]
                lin = self.plr_linear[i](x)
                freq = self.plr_freq[i]
                phase = self.plr_phase[i]
                fourier = torch.cat([
                    torch.sin(x * freq + phase),
                    torch.cos(x * freq + phase)
                ], dim=-1)
                fourier = self.plr_fourier_proj[i](fourier)
                plr = lin + fourier

                # Apply masking
                if mask is not None:
                    m = mask[:, i:i+1]  # [B, 1] — broadcasts correctly with [B, embed_dim]
                    emb = emb * m
                    plr = plr * m

                type0 = self.type_embed(torch.zeros(B, dtype=torch.long, device=cat_idx.device))
                type1 = self.type_embed(torch.ones(B, dtype=torch.long, device=cat_idx.device))
                tokens.append(emb + type0)
                tokens.append(plr + type1)

            # Budget tokens
            budget_proj = self.numeric_proj(budget_numeric)
            budget_proj = budget_proj.view(B, 2, -1)
            type2 = self.type_embed(torch.full((B,), 2, dtype=torch.long, device=cat_idx.device))
            for t in range(2):
                tokens.append(budget_proj[:, t] + type2)

            token_stack = torch.stack(tokens, dim=1)
            out = self.transformer(token_stack)
            out = out.view(B, -1)
            return self.head(out).squeeze(-1)

    class EMA:
        def __init__(self, model, decay=0.999):
            self.decay = decay
            self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

        def update(self, model):
            for k, v in model.state_dict().items():
                self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.detach()

        def state_dict(self):
            return {k: v.clone() for k, v in self.shadow.items()}

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train[TARGET])):
        t0 = time.time()
        print(f"  NN fold {fold+1}/{n_folds}")

        train_df = train.iloc[tr_idx].reset_index(drop=True)
        val_df = train.iloc[va_idx].reset_index(drop=True)
        test_df = test.reset_index(drop=True)

        def df_to_tensors(df, is_training=False):
            n = len(df)
            cat_idx = np.zeros((n, len(NUM_COLS)), dtype=np.int64)
            numeric_vals = np.zeros((n, len(NUM_COLS)), dtype=np.float32)

            for i, col in enumerate(NUM_COLS):
                vocab = vocab_maps[col]
                col_vals = df[col].values
                # Round to 2 decimals to match vocab keys (quantization)
                col_vals_rounded = np.round(col_vals, 2)
                # Vectorized: map known values, NaN stays at 0
                is_nan = pd.isna(col_vals)
                # Map non-NaN values through vocab (vectorized via pandas map)
                mapped = pd.Series(col_vals_rounded).map(vocab).fillna(0).astype(np.int64).values
                cat_idx[:, i] = mapped
                cat_idx[is_nan, i] = 0  # NaN → index 0
                # Numeric: fill NaN with 0, use raw (unrounded) values for PLR
                numeric_vals[:, i] = np.where(is_nan, 0.0, col_vals.astype(np.float32))

            budget_features = df[budget_cols + missing_cols].fillna(0).values.astype(np.float32)

            mask = np.ones((n, len(NUM_COLS)), dtype=np.float32)
            if is_training:
                mask_rand = np.random.random(mask.shape)
                mask[mask_rand < 0.1] = 0.0

            if TARGET in df.columns:
                labels = df[TARGET].values.astype(np.float32)
            else:
                labels = np.zeros(n, dtype=np.float32)  # test set has no target

            return (
                torch.LongTensor(cat_idx),
                torch.FloatTensor(numeric_vals),
                torch.FloatTensor(budget_features),
                torch.FloatTensor(mask),
                torch.FloatTensor(labels)
            )

        tr_tensors = df_to_tensors(train_df, is_training=True)
        va_tensors = df_to_tensors(val_df)
        te_tensors = df_to_tensors(test_df)

        train_ds = TensorDataset(*tr_tensors)
        train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0,
                                  pin_memory=(device.type == 'cuda'))

        vocab_sizes = [len(vocab_maps[col]) for col in NUM_COLS]
        model = EmbeddingPLRTransformer(
            vocab_sizes=vocab_sizes, embed_dim=16, n_heads=4,
            n_layers=2, plr_frequencies=16, dropout=0.1
        ).to(device)

        ema = EMA(model, decay=0.999)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=1e-3, epochs=epochs, steps_per_epoch=len(train_loader)
        )
        criterion = nn.BCEWithLogitsLoss()

        best_val_auc = 0
        best_epoch = 0
        best_oof_preds = None
        best_test_preds = None

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                cat_idx_b, num_vals_b, budget_b, mask_b, labels_b = [x.to(device) for x in batch]

                # Re-apply random masking per batch
                mask_b = torch.ones_like(mask_b)
                rand_mask = torch.rand_like(mask_b)
                mask_b[rand_mask < 0.1] = 0.0

                optimizer.zero_grad()
                logits = model(cat_idx_b, num_vals_b, budget_b, mask_b)
                loss = criterion(logits, labels_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                ema.update(model)

            # Validate every 4 epochs or at end
            if (epoch + 1) % 4 == 0 or epoch == epochs - 1:
                orig_state = {k: v.clone() for k, v in model.state_dict().items()}
                model.load_state_dict(ema.state_dict())
                model.eval()

                with torch.no_grad():
                    va_logits = []
                    for i in range(0, len(val_df), 2048):
                        cat_b = va_tensors[0][i:i+2048].to(device)
                        num_b = va_tensors[1][i:i+2048].to(device)
                        bud_b = va_tensors[2][i:i+2048].to(device)
                        msk_b = va_tensors[3][i:i+2048].to(device)
                        logits = model(cat_b, num_b, bud_b, msk_b)
                        va_logits.append(logits.cpu().numpy())
                    va_preds = 1 / (1 + np.exp(-np.concatenate(va_logits)))
                    va_auc = roc_auc_score(val_df[TARGET], va_preds)

                if va_auc > best_val_auc:
                    best_val_auc = va_auc
                    best_epoch = epoch + 1
                    with torch.no_grad():
                        te_logits = []
                        for i in range(0, len(test_df), 2048):
                            cat_b = te_tensors[0][i:i+2048].to(device)
                            num_b = te_tensors[1][i:i+2048].to(device)
                            bud_b = te_tensors[2][i:i+2048].to(device)
                            msk_b = te_tensors[3][i:i+2048].to(device)
                            logits = model(cat_b, num_b, bud_b, msk_b)
                            te_logits.append(logits.cpu().numpy())
                        best_test_preds = 1 / (1 + np.exp(-np.concatenate(te_logits)))
                    best_oof_preds = va_preds.copy()

                model.load_state_dict(orig_state)
                print(f"    Epoch {epoch+1}/{epochs} — Val AUC: {va_auc:.6f} (best: {best_val_auc:.6f})")

        oof[va_idx] = best_oof_preds
        test_preds += best_test_preds / n_folds
        fold_aucs.append(best_val_auc)

        elapsed = time.time() - t0
        print(f"    Fold {fold+1} best AUC: {best_val_auc:.6f} (epoch {best_epoch}, {elapsed:.1f}s)")

    overall_auc = roc_auc_score(train[TARGET], oof)
    print(f"  NN OOF AUC: {overall_auc:.6f}")
    return oof, test_preds, {'overall': overall_auc, 'folds': fold_aucs}


# ═══════════════════════════════════════════════════════════════════════════
# 5. RANK-NORMALIZED BLEND
# ═══════════════════════════════════════════════════════════════════════════

def rank_normalize(preds):
    return rankdata(preds) / len(preds)


def blend_models(oof_dict, test_dict, weights=None):
    if weights is None:
        weights = {'nn': 0.49, 'catboost': 0.31, 'lightgbm': 0.21}

    oof_blend = sum(weights[k] * rank_normalize(v) for k, v in oof_dict.items())
    test_blend = sum(weights[k] * rank_normalize(v) for k, v in test_dict.items())
    return oof_blend, test_blend


# ═══════════════════════════════════════════════════════════════════════════
# 6. MAIN
# ═══════════════════════════════════════════════════════════════════════════

def main():
    t_start = time.time()
    print("=" * 70)
    print("Sprint 5 — Omralinov-inspired pipeline (Kaggle GPU)")
    print("=" * 70)

    # Load data
    train, test, sub = load_data()
    print(f"Train: {train.shape}, Test: {test.shape}")

    # Verify budget constraint on complete rows
    complete = train[['daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours']].notna().all(axis=1)
    slack = train.loc[complete, 'daily_screen_time_hours'] - train.loc[complete, ['social_media_hours', 'gaming_hours', 'work_study_hours']].sum(axis=1)
    print(f"Budget constraint (complete rows): {(slack >= -1e-10).all()}, min slack: {slack.min():.4f}")

    # Feature engineering
    train, test = prepare_data(train, test)

    n_folds = 11
    seed = 42

    # Train all 3 models
    print("\n── CatBoost ──")
    cb_oof, cb_test, cb_metrics = train_catboost(train, test, n_folds=n_folds, seed=seed)

    print("\n── LightGBM ──")
    lgb_oof, lgb_test, lgb_metrics = train_lightgbm(train, test, n_folds=n_folds, seed=seed)

    print("\n── Neural Net (Embedding + PLR + Transformer) ──")
    nn_oof, nn_test, nn_metrics = train_neural_net(train, test, n_folds=n_folds, seed=seed, epochs=32)

    # Blend
    print("\n── Blending ──")
    oof_dict = {'nn': nn_oof, 'catboost': cb_oof, 'lightgbm': lgb_oof}
    test_dict = {'nn': nn_test, 'catboost': cb_test, 'lightgbm': lgb_test}

    # Default Omralinov weights
    oof_blend, test_blend = blend_models(oof_dict, test_dict,
                                          weights={'nn': 0.49, 'catboost': 0.31, 'lightgbm': 0.21})
    blend_auc = roc_auc_score(train[TARGET], oof_blend)
    print(f"Blend OOF AUC (0.49/0.31/0.21): {blend_auc:.6f}")

    # Also try equal weights
    oof_eq, test_eq = blend_models(oof_dict, test_dict,
                                    weights={'nn': 1/3, 'catboost': 1/3, 'lightgbm': 1/3})
    eq_auc = roc_auc_score(train[TARGET], oof_eq)
    print(f"Blend OOF AUC (equal 1/3):      {eq_auc:.6f}")

    # Create submission (use best blend)
    best_weights = {'nn': 0.49, 'catboost': 0.31, 'lightgbm': 0.21} if blend_auc >= eq_auc else {'nn': 1/3, 'catboost': 1/3, 'lightgbm': 1/3}
    best_oof = oof_blend if blend_auc >= eq_auc else oof_eq
    best_test = test_blend if blend_auc >= eq_auc else test_eq
    best_auc = max(blend_auc, eq_auc)

    sub[TARGET] = best_test
    sub.to_csv("submission.csv", index=False)

    # Save all predictions for potential re-blending
    np.save("oof_nn.npy", nn_oof)
    np.save("oof_catboost.npy", cb_oof)
    np.save("oof_lightgbm.npy", lgb_oof)
    np.save("oof_blend.npy", best_oof)
    np.save("test_nn.npy", nn_test)
    np.save("test_catboost.npy", cb_test)
    np.save("test_lightgbm.npy", lgb_test)

    # Summary
    elapsed = time.time() - t_start
    print(f"\n{'='*70}")
    print(f"FINAL RESULTS (Sprint 5 — Omralinov-inspired)")
    print(f"{'='*70}")
    print(f"CatBoost  OOF AUC: {cb_metrics['overall']:.6f}")
    print(f"LightGBM  OOF AUC: {lgb_metrics['overall']:.6f}")
    print(f"NN        OOF AUC: {nn_metrics['overall']:.6f}")
    print(f"Blend     OOF AUC: {best_auc:.6f}")
    print(f"Blend weights: {best_weights}")
    print(f"Total time: {elapsed/60:.1f} min")
    print(f"Target: beat autonomous 0.96970 → need +0.0008 minimum")


if __name__ == "__main__":
    main()


---
## Results

| Model | Local AUC (2 folds, 1 epoch NN) | Notes |
|-------|--------------------------------|-------|
| CatBoost | 0.9434 | CPU test, 2 folds |
| LightGBM | 0.9440 | 2 folds, target encoding |
| Neural Net | ~0.49 | 1 epoch smoke test — 32 epochs on Kaggle GPU |
| **Blend** | **0.817** | Rank-normalized OOF blend |

On Kaggle with 11 folds, 32 epochs NN, and GPU: expected LB ~0.970+